# Standardized Layer Exploration

In [ ]:
import polars as pl
import duckdb
import pathlib
from datetime import date

# Paths
DATA_ROOT = pathlib.Path("../data")
STANDARDIZED_ROOT = DATA_ROOT / "standardized"

print("Polars version:", pl.__version__)
print("DuckDB version:", duckdb.__version__)

Polars version: 1.40.1
DuckDB version: 1.5.2


#### Cell: **List all available partitions**

In [2]:
# List all standardized datasets
datasets = [p.name for p in STANDARDIZED_ROOT.iterdir() if p.is_dir()]
print("Available datasets:", datasets)

# Show partitions for each
for ds in datasets:
    partitions = list((STANDARDIZED_ROOT / ds).glob("date=*/"))
    dates = sorted([p.name.split("=")[1] for p in partitions])
    print(f"{ds:25} → {len(dates)} partitions | {dates[:3]} ...")

Available datasets: ['gps_positions', 'gtfs_routes', 'gtfs_stops', 'gtfs_stop_times', 'gtfs_trips']
gps_positions             → 7 partitions | ['2026-05-12', '2026-05-13', '2026-05-14'] ...
gtfs_routes               → 7 partitions | ['2026-05-12', '2026-05-13', '2026-05-14'] ...
gtfs_stops                → 7 partitions | ['2026-05-12', '2026-05-13', '2026-05-14'] ...
gtfs_stop_times           → 7 partitions | ['2026-05-12', '2026-05-13', '2026-05-14'] ...
gtfs_trips                → 7 partitions | ['2026-05-12', '2026-05-13', '2026-05-14'] ...


In [9]:
con = duckdb.connect()

# GPS positions example
query = """
SELECT 
    snapshot_ts,
    transport_type,
    line_number,
    lon,
    lat,
    speed_kmh,
    vehicle_id
FROM '../data/standardized/gps_positions/date=*/part.parquet'
WHERE line_number = '8'
LIMIT 10
"""

df = con.sql(query).pl()   # DuckDB → Polars
print(df)

shape: (10, 7)
┌─────────────────┬────────────────┬─────────────┬──────────┬──────────┬───────────┬────────────┐
│ snapshot_ts     ┆ transport_type ┆ line_number ┆ lon      ┆ lat      ┆ speed_kmh ┆ vehicle_id │
│ ---             ┆ ---            ┆ ---         ┆ ---      ┆ ---      ┆ ---       ┆ ---        │
│ datetime[μs,    ┆ i32            ┆ str         ┆ f64      ┆ f64      ┆ i32       ┆ str        │
│ Europe/Tallinn] ┆                ┆             ┆          ┆          ┆           ┆            │
╞═════════════════╪════════════════╪═════════════╪══════════╪══════════╪═══════════╪════════════╡
│ 2026-05-12      ┆ 2              ┆ 8           ┆ 24.65884 ┆ 59.41081 ┆ null      ┆ 1022       │
│ 21:07:22 EEST   ┆                ┆             ┆          ┆          ┆           ┆            │
│ 2026-05-12      ┆ 2              ┆ 8           ┆ 24.65842 ┆ 59.41046 ┆ null      ┆ 1024       │
│ 21:07:22 EEST   ┆                ┆             ┆          ┆          ┆           ┆            │
│ 202